# NORIA — HQ talking clips (SadTalker + working GFPGAN enhancer)

Fixes the enhancer properly by pinning **torchvision 0.15.2** (which still has
`functional_tensor`, the thing basicsr needs — no hacks). Then: square image
prep (no stretch), GFPGAN face enhancement, 512px, tuned movement, and H.264
output for clean browser playback.

Honest ceiling: this is the **best SadTalker can do** — much sharper and more
natural than before, but SadTalker is a 2023 model, not indistinguishable-human.

**Run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

## 0. GPU on?

In [ ]:
!nvidia-smi -L

## 1. Python 3.11 env + SadTalker

In [ ]:
!pip -q install uv
!uv python install 3.11
%cd /content
!git clone -q https://github.com/OpenTalker/SadTalker || echo "already cloned"
%cd /content/SadTalker
import os
if not os.path.isdir('.venv'):
    get_ipython().system('uv venv --python 3.11 .venv')
else:
    print('.venv exists — keeping')
print('env ready')

## 2. Install COMPATIBLE versions (torchvision 0.15.2 → enhancer works natively)

In [ ]:
# torch/vision the enhancer stack was built for (0.15.2 still has functional_tensor)
!uv pip install --python .venv/bin/python torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!uv pip install --python .venv/bin/python "setuptools<70" wheel
# numpy<1.24 (SadTalker ragged arrays) + a matching scientific stack
!uv pip install --python .venv/bin/python "numpy==1.23.5" "scipy==1.10.1" "scikit-image==0.21.0" "numba==0.57.1" "llvmlite==0.40.0" "opencv-python==4.9.0.80" "librosa==0.10.1" resampy soundfile imageio imageio-ffmpeg kornia yacs pydub safetensors av tqdm edge-tts pillow
# enhancer deps (build against the env, not a fresh isolated one)
!uv pip install --python .venv/bin/python --no-build-isolation basicsr facexlib gfpgan "face-alignment==1.3.5"
# keep numpy pinned
!uv pip install --python .venv/bin/python "numpy==1.23.5"

## 3. Compatibility check + download models
This must print SUCCESS — it means the GFPGAN enhancer will work.

In [ ]:
!.venv/bin/python -c "from basicsr.data.degradations import circular_lowpass_kernel; print('SUCCESS: BasicSR enhancer dependency is working')"
!bash scripts/download_models.sh

## 4. Voice — clean 24 kHz WAV (edit the line)

In [ ]:
NORIA_LINE = "Hello, I'm Noria, your SkyGlobe companion. It's really good to finally meet you."
!.venv/bin/edge-tts --voice en-US-JennyNeural --rate=-4% --text "{NORIA_LINE}" --write-media /content/f.mp3
!.venv/bin/edge-tts --voice en-US-GuyNeural   --rate=-4% --text "{NORIA_LINE}" --write-media /content/m.mp3
!ffmpeg -y -loglevel error -i /content/f.mp3 -ar 24000 -ac 1 /content/f.wav
!ffmpeg -y -loglevel error -i /content/m.mp3 -ar 24000 -ac 1 /content/m.wav
print('voices ready')

## 5. Get faces + prepare them 512×512 (pad, do NOT stretch)

In [ ]:
!wget -q -O /content/noria-f.png https://noria-body.onrender.com/assets/noria-f.png
!wget -q -O /content/noria-m.png https://noria-body.onrender.com/assets/noria-m.png
from PIL import Image
for name in ['noria-f', 'noria-m']:
    im = Image.open(f'/content/{name}.png').convert('RGB')
    w, h = im.size; s = max(w, h)
    bg = Image.new('RGB', (s, s), (18, 28, 42))  # pad to square (no vertical stretch)
    bg.paste(im, ((s - w) // 2, (s - h) // 2))
    bg.resize((512, 512), Image.LANCZOS).save(f'/content/{name}_512.png')
print('prepped 512x512')

## 6. Render HQ (GFPGAN enhancer, still, subtle expression) — a few min each

In [ ]:
%cd /content/SadTalker
!.venv/bin/python inference.py --driven_audio /content/f.wav --source_image /content/noria-f_512.png --result_dir /content/out_f --still --preprocess full --size 512 --expression_scale 0.9 --enhancer gfpgan
!.venv/bin/python inference.py --driven_audio /content/m.wav --source_image /content/noria-m_512.png --result_dir /content/out_m --still --preprocess full --size 512 --expression_scale 0.9 --enhancer gfpgan
print('rendered')

## 7. Re-encode H.264 + watch + download

In [ ]:
import glob, os
from IPython.display import HTML, display
from base64 import b64encode
def finish(tag, d, out):
    mp4s = sorted([m for m in glob.glob(d + '/*.mp4')], key=os.path.getmtime)
    if not mp4s: print('NO VIDEO for', tag, '— see render output above'); return
    src = mp4s[-1]
    os.system(f'ffmpeg -y -loglevel error -i "{src}" -c:v libx264 -crf 18 -preset medium -pix_fmt yuv420p -movflags +faststart {out}')
    print(tag, '->', out)
    data = b64encode(open(out, 'rb').read()).decode()
    display(HTML(f'<b>{tag}</b><br><video width=420 controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))
finish('Noria-F', '/content/out_f', '/content/noria_f_HQ.mp4')
finish('Noria-M', '/content/out_m', '/content/noria_m_HQ.mp4')
try:
    from google.colab import files; files.download('/content/noria_f_HQ.mp4'); files.download('/content/noria_m_HQ.mp4')
except Exception:
    print('Download from Files panel: /content/noria_f_HQ.mp4 and /content/noria_m_HQ.mp4')